# ICU Length of Stay Prediction Demo

This notebook demonstrates the final capstone workflow using a small synthetic sample. The real project uses restricted ICU data, so this public demo imports precomputed sample predictions and summary outputs instead of retraining on the full dataset.

## Setup

Load the sample files included in `data/sample`. The notebook is designed to run in under 1 minute.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd()
if not (ROOT / "data" / "sample").exists():
    ROOT = ROOT.parent
SAMPLE_DIR = ROOT / "data" / "sample"


In [ ]:
predictions = pd.read_csv(SAMPLE_DIR / "demo_predictions.csv")
metrics = pd.read_csv(SAMPLE_DIR / "demo_model_metrics.csv")
feature_importance = pd.read_csv(SAMPLE_DIR / "demo_feature_importance.csv")
subgroup_error = pd.read_csv(SAMPLE_DIR / "demo_subgroup_error.csv")

predictions.head()


In [ ]:
def bar_chart(df, label_col, value_col, title, color="#4c78a8"):
    rows = df[[label_col, value_col]].copy()
    max_value = rows[value_col].max()
    html = [f"<h3>{title}</h3>", "<div style='font-family: sans-serif; max-width: 760px'>"]
    for _, row in rows.iterrows():
        width = 100 * row[value_col] / max_value if max_value else 0
        html.append(
            f"<div style='display:grid; grid-template-columns: 230px 1fr 60px; gap:8px; align-items:center; margin:6px 0'>"
            f"<div>{row[label_col]}</div>"
            f"<div style='background:#edf1f4; height:18px'><div style='background:{color}; width:{width:.1f}%; height:18px'></div></div>"
            f"<div>{row[value_col]:.3g}</div></div>"
        )
    html.append("</div>")
    return HTML("".join(html))

def prediction_scatter(df):
    x = df["observed_los_days"]
    y = df["predicted_los_days"]
    limit = max(x.max(), y.max())
    points = []
    for observed, predicted in zip(x, y):
        px = 40 + 320 * observed / limit
        py = 360 - 320 * predicted / limit
        points.append(f"<circle cx='{px:.1f}' cy='{py:.1f}' r='4' fill='#4c78a8' opacity='0.75'/>")
    svg = f"""
    <h3>Predicted vs. observed ICU LOS</h3>
    <svg width="420" height="410" viewBox="0 0 420 410" role="img">
      <rect x="0" y="0" width="420" height="410" fill="white"/>
      <line x1="40" y1="360" x2="360" y2="40" stroke="#555" stroke-dasharray="5,5"/>
      <line x1="40" y1="360" x2="370" y2="360" stroke="#222"/>
      <line x1="40" y1="360" x2="40" y2="30" stroke="#222"/>
      {"".join(points)}
      <text x="160" y="400" font-size="13">Observed LOS, days</text>
      <text x="8" y="210" font-size="13" transform="rotate(-90 8 210)">Predicted LOS, days</text>
    </svg>
    """
    return HTML(svg)


## Model Comparison

The full project compared accelerated failure time survival models with a log-transformed length-of-stay gradient boosting model.

In [ ]:
metrics


In [ ]:
display(bar_chart(metrics, "model", "test_mae_days", "Test MAE by model", color="#f58518"))


## Example Predictions

These synthetic examples show cases where the final model is close and cases where ICU length of stay is harder to predict.

In [ ]:
display_cols = ["patient_id", "age_group", "careunit_group", "observed_los_days", "predicted_los_days", "absolute_error_days", "example_type"]
examples = pd.concat([
    predictions.nsmallest(5, "absolute_error_days"),
    predictions.nlargest(5, "absolute_error_days"),
])[display_cols]
examples


In [ ]:
display(prediction_scatter(predictions))


## Error By Care Unit

Subgroup diagnostics help identify where predictions are less reliable.

In [ ]:
subgroup_error.sort_values("mae_days", ascending=False)


In [ ]:
display(bar_chart(subgroup_error.sort_values("mae_days"), "careunit_group", "mae_days", "Synthetic demo error by care unit", color="#54a24b"))


## Interpretation

The project uses early ICU information including demographics, care unit, first-day vitals, labs, input events, and missingness patterns.

In [ ]:
display(bar_chart(feature_importance.sort_values("relative_importance"), "feature", "relative_importance", "Feature groups used by the final model", color="#b279a2"))


## Takeaways

- ICU length of stay is right-skewed, so log-scale modeling and robust error summaries are important.
- The gradient boosting log-LOS model had the best overall discrimination and mean absolute error in the final comparison.
- Very long ICU stays remain difficult to predict from first-day data alone, so subgroup and tail diagnostics are essential.